# VLM-DENTAL — External Benchmarks (MMOral-OPG-Bench)

This notebook measures the **final checkpoint** (GRPO trained with the rules + skills harness) on a benchmark we did not build, and puts it next to the **raw Qwen3.5-9B** and to public dental models, **all scored by the same judge**. It trains nothing, tunes nothing, and never touches the DENTEX test split.

### What it runs
| Row of the paper table | What happens | Phase |
|---|---|---|
| **Qwen3.5-9B raw** | image + the benchmark's official prompt, answered directly by the untrained base model | 2a |
| **GRPO + harness, agent-assisted (Option A)** | the trained agent analyses each radiograph once with its native prompt and the frozen protocol; its findings are put in front of the same official prompt; the **same raw Qwen3.5-9B answers** | 1, 2b |
| **OralGPT-Omni-7B, OralGPT-OPG-7B** (public releases) | the same official prompt, their own weights, our judge | 2c |
| Published numbers of other papers | typed in by hand with source and judge, printed as separate rows | 4 |

Because the raw row and the Option A row use the *same answering model and prompt*, their paired difference is exactly what the agent's findings add. Option B (the agent answers in free text with its own prompt) is **not** run: the adapter was never trained on that answer format. It is the fallback if Option A is low.

### Benchmark
**MMOral-OPG-Bench** (NeurIPS 2025): 100 panoramic radiographs, **491 closed-ended** (4 options, exact-match accuracy) and **578 open-ended** questions (LLM-judged, 0-1) in five dimensions: Teeth, Pathology (Patho), History of treatment (HisT), Jaw, Summary and recommendation (SumRec). Public, MIT licence, no gate. The prompts, the answer extraction, the judge prompt and the 5-try rule are ported verbatim from the benchmark authors' EvalKit.

**Not run, and why:** MMOral-Uni-Bench (gated; its modalities are intraoral photos, periapical and cephalometric films, pathology images and video, none panoramic), MMOral-OPG-Training (training data), OralDetect-Bench (detection, not VQA), MMOral-X and OPG-Bench (no public release found). See `docs/EXTERNAL_BENCHMARKS.md`.

### Invariants
- One hardware profile per run (`HARDWARE_PROFILE`, the same default as the other notebooks); the results folder carries the profile tag, and the policy checkpoint must have been trained under the same profile. No profile crossing.
- The policy runs under the **frozen evaluation protocol** (50 tool calls, 16384 new tokens, canonical views, greedy). Nothing is lowered.
- A change of any setting that decides what a number means (judge, token budget, adapter, prompt version, findings source) makes a resume **refuse**; nothing is mixed.
- The judge is a **setting**, recorded in every judged row. Cross-paper numbers are not comparable unless the judge matches; this notebook never mixes them.
- Freeze the checkpoint, K and settings **before** the first full run. A change after seeing scores is disclosed in the paper.
- Every phase is resumable and the results folder can be synced to the Hub (Phase 8).

## 1. Platform Detection & Repository Root
Finds the repository root (case-insensitively: the folder may be `VLM-DENTAL` or `VLM-Dental`) before ever cloning.

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
PLATFORM_NAME = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local PC / Server")
print(f"[PLATFORM] Detected Runtime Environment: {PLATFORM_NAME}")

REPO_NAME = "VLM-DENTAL"
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if p.name.lower() == REPO_NAME.lower()), None)
if repo_root is None:
    repo_root = next((c for c in cwd.iterdir() if c.is_dir() and c.name.lower() == REPO_NAME.lower()), None)

if repo_root is None:
    print(f"[CLONE] No local repo found. Shallow cloning {REPO_URL}...")
    !git clone --depth 1 {REPO_URL}
    repo_root = cwd / REPO_NAME
    if not repo_root.is_dir():
        raise RuntimeError("git clone failed, check network / URL / permissions")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"[WORKSPACE] Using repository root: {Path.cwd()}")

## 2. Dependencies
The environment is the one built by `scripts/setup_gpu_server.sh` (see `docs/RUNBOOK_SFT_GRPO.md` A0): `requirements.txt` already pins everything this notebook uses, including `openai` for the judge, and torch is pinned by the setup script. **This notebook installs nothing**, so no pin can move. The audit cell imports every module the phases use, so a missing package stops here, not in the middle of a run.

In [ ]:
import importlib

AUDIT = [
    ("torch", "torch"), ("transformers (>=5 for Qwen3.5)", "transformers"), ("peft", "peft"), ("huggingface_hub", "huggingface_hub"),
    ("openai (judge)", "openai"), ("pandas", "pandas"), ("PIL", "PIL"), ("tabulate", "tabulate"), ("dotenv", "dotenv"),
    ("repo: agent loop", "dental_agent.agent.loop"), ("repo: local runtime", "dental_agent.model.local_runtime"),
    ("repo: evaluator", "scripts.evaluate_models"), ("repo: tool registry", "dental_agent.tools.registry"),
    ("repo: harness", "dental_agent.harness.notebook_eval"), ("repo: benchmarks", "dental_agent.benchmarks.mmoral_opg"),
]
missing = []
for label, module in AUDIT:
    try:
        mod = importlib.import_module(module)
        print(f"[OK]      {label:<34} {getattr(mod, '__version__', '')}")
    except Exception as error:                                     # noqa: BLE001 - reported, then raised below
        missing.append((label, module, repr(error)))
        print(f"[MISSING] {label:<34} {error!r}")
if missing:
    raise RuntimeError("Fix these before running anything (re-run scripts/setup_gpu_server.sh or the SFT notebook install cell): "
                       + ", ".join(m[0] for m in missing))

## 3. Hardware
Checks the GPU against the selected profile with the same function the evaluation notebooks use (`hardware_reason`): native bf16 needs Ampere or newer, and an unquantized 9B needs about 18 GB before activations. The profile itself is chosen in the settings cell (section 6).

In [ ]:
import torch

DEVICES = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        DEVICES.append(dict(name=prop.name, gib=prop.total_memory / 2**30, capability=torch.cuda.get_device_capability(i), tpu=False))
        print(f"[HARDWARE] cuda:{i} {prop.name}  {prop.total_memory / 2**30:.1f} GiB  capability {torch.cuda.get_device_capability(i)}")
else:
    print("[HARDWARE] No CUDA device: only the CPU-only phases (prepare, leakage, judge, score) can run.")
print(f"[FRAMEWORK] PyTorch {torch.__version__}")

## 4. Secrets, Hub Authentication
`HF_TOKEN` (read access to the gated model pages you accepted; write access for Phase 8), `HF_ARTIFACT_REPO` (trained checkpoints), `DENTEX_IMAGES_REPO` (for the leakage screen), and the judge key. The key variable is named in the settings cell (`JUDGE_KEY_ENV`), so any OpenAI-compatible endpoint works.

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login
from tabulate import tabulate

load_dotenv()

def get_secret_multisource(key, default=None):
    val = os.environ.get(key)
    if val:
        return val, ".env / OS Env"
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            sec = UserSecretsClient().get_secret(key)
            if sec:
                os.environ[key] = sec
                return sec, "Kaggle Secrets"
        except Exception:
            pass
    if IS_COLAB:
        try:
            from google.colab import userdata
            sec = userdata.get(key)
            if sec:
                os.environ[key] = sec
                return sec, "Colab Userdata"
        except Exception:
            pass
    if default is not None:
        os.environ[key] = default
        return default, "Default"
    return None, "Not Set"

def mask_secret(val, is_secret=True):
    if not val:
        return "---"
    if not is_secret:
        return val
    return "********" if len(val) <= 8 else f"{val[:4]}...{val[-4:]}"

TOKEN_SPECS = [
    ("HF_TOKEN", True, None),
    ("HF_ARTIFACT_REPO", False, "Reza-Nadimi/vlm-dental-models"),
    ("DENTEX_IMAGES_REPO", False, "Reza-Nadimi/dentex-train-images"),
    ("OPENAI_API_KEY", True, None),
    ("DENTAL_AGENT_DATA_DIR", False, "data"),
]
rows = []
for key, is_sec, default in TOKEN_SPECS:
    val, source = get_secret_multisource(key, default)
    rows.append([key, "[SET]" if val else "[NOT SET]", source, mask_secret(val, is_sec)])
print(tabulate(rows, headers=["Environment Variable", "Status", "Source", "Value"], tablefmt="fancy_grid"))

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("[AUTH] Authenticated with the Hugging Face Hub.")
else:
    raise RuntimeError("HF_TOKEN is not set: checkpoints and gated models cannot be fetched.")

## 5. Gated-Access Audit
The benchmark itself is public. The three OralGPT model pages are **gated**: open each link once with the account that owns `HF_TOKEN` and click **Agree**. This cell reports which ones your token can read, so a refusal appears here and not after the GPU is already busy.

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.errors import GatedRepoError, RepositoryNotFoundError

api = HfApi()
ACCESS_CHECKS = [
    ("MMOral-OPG-Bench (public, MIT)", "OralGPT/MMOral-OPG-Bench", "dataset"),
    ("OralGPT-Omni-7B-Instruct (gated)", "OralGPT/OralGPT-Omni-7B-Instruct", "model"),
    ("OralGPT-OPG-7B-Instruct (gated)", "OralGPT/OralGPT-OPG-7B-Instruct", "model"),
    ("OralGPT-Plus-7B (gated, optional)", "OralGPT/OralGPT-Plus-7B", "model"),
]
GATE_STATUS = {}
for label, repo, kind in ACCESS_CHECKS:
    try:
        api.auth_check(repo_id=repo, repo_type=kind)
        GATE_STATUS[repo] = True
        print(f"[ACCESS OK]      {label}")
    except GatedRepoError:
        GATE_STATUS[repo] = False
        base = "datasets/" if kind == "dataset" else ""
        print(f"[NOT ACCEPTED]   {label}: open https://huggingface.co/{base}{repo} and click Agree")
    except RepositoryNotFoundError:
        GATE_STATUS[repo] = False
        print(f"[NOT FOUND]      {label}: repository missing or hidden from this token")

## 6. Settings (the control centre)
Every value that changes a result is visible here; nothing is chosen automatically. `EVAL_PROFILES` is the same block as in the Final_Evaluation notebook and must stay identical across notebooks (it is part of every result's identity).

- **Policy** (`POLICY_*`): the exact trained checkpoint; K and harness mode must be the ones you froze for the paper.
- **`MAX_NEW_TOKENS = 16384`**: used for the policy (frozen protocol) and for every answering model, so no system is cut shorter than another.
- **Judge**: `JUDGE_MODEL` has **no default**. For reference, the benchmark authors' own config uses `gpt-5-mini` and OralGPT-Plus's scorer defaults to `gpt-5-nano`; the papers do not all use the same judge. Pick one, write it in the paper, and use it for every row. `JUDGE_SEND_TEMPERATURE = False` if the endpoint refuses the temperature parameter (reasoning models do).
- **`SMOKE_LIMIT`**: first N images / questions only. Smoke results are real results of those items and a full run resumes after them.

In [ ]:
import json
import re
import shlex
import subprocess

from dental_agent.harness.notebook_eval import runtime_flags, hardware_reason

# --- hardware profile (identical block to the Final_Evaluation notebook) -------------------------------------------------
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "quadro_rtx8000": {
        "tag": "quadro_rtx8000_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False,
        "serving_dtype": "float16"
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    }
}
HARDWARE_PROFILE = "quadro_rtx8000"                # quadro_rtx8000 | workstation48_lora | rtx4090_qlora | dual_t4 | ... (the default of the other notebooks)
PROFILE = EVAL_PROFILES[HARDWARE_PROFILE]
TAG = PROFILE["tag"]
RUNTIME_FLAGS = runtime_flags(HARDWARE_PROFILE, EVAL_PROFILES)
LETTERBOX_FLAG = "--canonical-letterbox" if PROFILE["letterbox"] else "--no-canonical-letterbox"

# --- paths ---------------------------------------------------------------------------------------------------------------
DATA_DIR = os.environ.get("DENTAL_AGENT_DATA_DIR", "data")
BENCH_DIR = f"{DATA_DIR}/benchmarks/mmoral_opg"
RESULTS_DIR = f"{DATA_DIR}/evaluations/external/mmoral_opg/{TAG}"          # one folder per hardware profile tag
PUBLISHED_NUMBERS = f"{BENCH_DIR}/published_numbers.json"                  # typed in by hand (Phase 4)
HF_ARTIFACT_REPO = os.environ["HF_ARTIFACT_REPO"]
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"                   # existing; never created by code
EVAL_RESULTS_REPO_TYPE = None                                              # None: detect the existing model/dataset repository

# --- models and budgets --------------------------------------------------------------------------------------------------
# BASE MODEL: ONE local folder, found by resolve_qwen_model() (nothing is copied or moved). As in the SFT and GRPO notebooks, the first call
# DOWNLOADS the model (about 19 GB) when no local copy exists; it prints where the weights come from.
FORCE_HF_MODEL_DOWNLOAD = False
from dental_agent.utils.inputs import LOCAL_BASE_MODEL_ENV, resolve_qwen_model
MODEL_ID, MODEL_SOURCE = resolve_qwen_model(force_download=FORCE_HF_MODEL_DOWNLOAD)
os.environ[LOCAL_BASE_MODEL_ENV] = MODEL_ID                                # child processes follow the same folder
MAX_NEW_TOKENS = 16384
MAX_TOOL_CALLS = 50
# Precision of the public baselines (they ignore the other runtime flags). The profile's precision by default. The OralGPT releases are bf16
# checkpoints: on a card without native bf16 (Turing) the profile precision is fp16, which can overflow in some vision-language models. Phase 6
# shows empty or garbled answers if it does; the explicit alternative is "fp32" (about 32 GB of weights for an 8B model).
EXTERNAL_PRECISION = PROFILE["precision"]

# --- the trained policy under test (exact checkpoint, never substituted) ----------------------------------------------------
POLICY_TRACK = "with_tools"
POLICY_STAGE = "dentex_alone"
POLICY_K = 4
POLICY_HARNESS_MODE = "rules_skills_index"

# --- systems (names become folder names: letters, digits, _ . + - only) ---------------------------------------------------
POLICY_NAME = f"grpo_k{POLICY_K}_harness_{POLICY_HARNESS_MODE}"
SYSTEM_RAW = "qwen3_5_9b_raw"
SYSTEM_A = f"optionA_{POLICY_NAME}"
EXTERNAL_MODELS = {                                                        # public baselines: own weights, our judge
    "oralgpt_omni_7b": dict(repo="OralGPT/OralGPT-Omni-7B-Instruct", revision="", enabled=True),
    "oralgpt_opg_7b": dict(repo="OralGPT/OralGPT-OPG-7B-Instruct", revision="", enabled=True),
}

# --- what to run -----------------------------------------------------------------------------------------------------------
RUN_PREPARE = True
RUN_LEAKAGE = True
RUN_POLICY = True
RUN_RAW_BASELINE = True
RUN_OPTION_A = True
RUN_EXTERNAL_BASELINES = True
RUN_JUDGE = True
EXISTING_RESULTS = "resume"            # error | resume | overwrite
SMOKE_LIMIT = None                     # e.g. 2 = first 2 images / first 2 questions per kind; None = everything

# --- judge ---------------------------------------------------------------------------------------------------------------------
JUDGE_MODEL = None                     # REQUIRED before Phase 3, e.g. the same judge you cite in the paper
JUDGE_BASE_URL = ""                    # "" = the OpenAI endpoint; any OpenAI-compatible URL otherwise
JUDGE_KEY_ENV = "OPENAI_API_KEY"
JUDGE_SEND_TEMPERATURE = True
JUDGE_MAX_TOKENS = 16384
JUDGE_WORKERS = 4

# --- derived and checked --------------------------------------------------------------------------------------------------------
LIMIT_FLAGS = ["--limit", str(SMOKE_LIMIT)] if SMOKE_LIMIT else []
for name in (POLICY_NAME, SYSTEM_RAW, SYSTEM_A, *EXTERNAL_MODELS):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.+-]*", name):
        raise ValueError(f"Bad name {name!r}")
reason = hardware_reason(HARDWARE_PROFILE, "grpo", DEVICES, EVAL_PROFILES) if DEVICES else "pending: CUDA evaluation runtime required"
print(json.dumps(dict(profile=HARDWARE_PROFILE, tag=TAG, runtime_flags=RUNTIME_FLAGS, letterbox=LETTERBOX_FLAG, model_id=str(MODEL_ID), model_source=MODEL_SOURCE,
                      results_dir=RESULTS_DIR, policy=POLICY_NAME, systems=[SYSTEM_RAW, SYSTEM_A, *[k for k, v in EXTERNAL_MODELS.items() if v["enabled"]]],
                      judge=JUDGE_MODEL, smoke_limit=SMOKE_LIMIT, existing_results=EXISTING_RESULTS), indent=2))
print("[HARDWARE CHECK]", reason or "this device satisfies the selected profile")

def run_cmd(cmd, check=True):
    # Streams a child process into the cell. GPU phases run as child processes so every model load is freed when it ends.
    cmd = [str(c) for c in cmd]
    print("$ " + " ".join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    code_ = proc.wait()
    if check and code_:
        raise RuntimeError(f"command failed with exit code {code_}")
    return code_

def bench_cmd(sub, *extra):
    base = [sys.executable, "scripts/run_external_benchmark.py", sub, "--bench-dir", BENCH_DIR]
    if sub not in {"prepare", "leakage"}:
        base += ["--results-dir", RESULTS_DIR]
    return base + [str(e) for e in extra]

## 7. Phase 0a — Prepare the Benchmark
Downloads the two TSVs (382 MB, public), checks their **MD5 against the benchmark authors' values**, decodes every distinct radiograph once into `images/`, and writes slim question files. A file that differs from the authors' MD5 stops the step; re-run with `--allow-md5-mismatch` only after inspecting why, and the manifest then records it.

In [ ]:
if RUN_PREPARE:
    run_cmd(bench_cmd("prepare", "--download"))
manifest = json.loads(Path(BENCH_DIR, "prepared", "manifest.json").read_text())
print(f"\n{manifest['n_images']} radiographs | {manifest['n_closed']} closed | {manifest['n_open']} open | "
      f"MD5 matches the authors: {manifest['md5_matches_authors']}")
print(f"image sizes: min {manifest['image_size_min']}  max {manifest['image_size_max']}")
print(tabulate([[k, v] for k, v in sorted(manifest["closed_categories"].items())], headers=["closed category", "n"]))
print(tabulate([[k, v] for k, v in sorted(manifest["open_categories"].items())], headers=["open category", "n"]))
print("closed answer letters:", manifest["closed_answer_letters"])

In [ ]:
# One question of each kind with its image: a quick look that the adapter reads the benchmark the way you expect.
from IPython.display import display
from PIL import Image
from dental_agent.benchmarks import mmoral_opg as mm

closed_df, open_df, images_df = mm.load_prepared(Path(BENCH_DIR, "prepared"))
row = closed_df.iloc[0]
print(mm.closed_prompt(row["question"], [row[c] for c in mm.OPTION_COLUMNS]), "\n-> answer:", row["answer"], "| category:", row["category"])
orow = open_df.iloc[0]
print("\n" + mm.open_prompt(orow["question"]), "\n-> reference:", orow["answer"], "| category:", orow["category"])
img = Image.open(images_df.set_index("id").loc[int(row["image_id"]), "local_path"])
img.thumbnail((900, 450))
display(img)

## 8. Phase 0b — Leakage Screen against DENTEX
The policy was trained on DENTEX train images. This screen compares every benchmark radiograph with every DENTEX image: the combined train folders (about 1,294 images, which is what the YOLOv8m grounder was trained on and a superset of the VLM's training images) and the 50 validation images (difference hash, then cross-correlation of thumbnails; exact SHA-256 duplicates separately) and always lists the closest pairs so the margin is visible. It catches copies, rescaled copies and re-encodings, **not** crops, rotations or flips. If it finds an overlap the step stops: those images cannot be reported as held-out.

In [ ]:
if RUN_LEAKAGE:
    code_ = run_cmd(bench_cmd("leakage", "--data-dir", DATA_DIR), check=False)
    print("\n[LEAKAGE] exit code", code_, "(0 = no copy found; non-zero = overlap or an error: read the output above)")
report_path = Path(BENCH_DIR, "leakage_dentex.json")
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print(report["verdict"], "| closest pair: hamming", report["closest"][0]["hamming"], "ncc", round(report["closest"][0]["ncc"], 3))

## 9. Phase 1a — Fetch the Policy Checkpoint
Resolves the **exact** checkpoint by name from `experiment_profiles.grpo_name` (this server's `data/models` first, then the models Hub repository). A missing one raises; no other K, harness mode, stage or tag is substituted. The step metadata is printed so you can confirm it is the finished run (200 steps, harness mode) and not a probe.

In [ ]:
from dental_agent.benchmarks.policy_source import fetch_grpo_policy

POLICY = fetch_grpo_policy(HF_ARTIFACT_REPO, POLICY_TRACK, POLICY_K, POLICY_STAGE, PROFILE["precision"], TAG, POLICY_HARNESS_MODE,
                           local_models=f"{DATA_DIR}/models")
print("name   :", POLICY["name"])
print("source :", POLICY["source"])
print("adapter:", POLICY["path"])
print("meta   :", json.dumps(POLICY["meta"], indent=2)[:1500] if POLICY["has_meta"] else "no checkpoint_meta.json next to the adapter")

## 10. Phase 1b — The Agent Analyses Every Radiograph (GPU)
One episode per image with the **native** system prompt, the 8 tools, the harness (`rules_skills_index`), canonical views per the profile, greedy decoding, 50 tool calls, 16384 new tokens. Findings and the full trajectory are cached in `policy/<name>/findings.jsonl`. Tools run on the native image; `locate_tooth` is the DENTEX-trained YOLOv8m grounder, so this is also a test of the grounder on a different source of radiographs.

Nothing is truncated or retried: an episode with no valid final answer is recorded as such (`answered=false`), and Option A tells the answering model so, instead of pretending the image was healthy.

In [ ]:
policy_cmd = bench_cmd("policy", "--policy-name", POLICY_NAME, "--adapter-path", POLICY["path"], "--harness-mode", POLICY_HARNESS_MODE,
                       "--model-id", MODEL_ID, "--max-new-tokens", MAX_NEW_TOKENS, "--max-tool-calls", MAX_TOOL_CALLS,
                       "--existing-results", EXISTING_RESULTS, LETTERBOX_FLAG, *RUNTIME_FLAGS, *LIMIT_FLAGS)
if RUN_POLICY:
    run_cmd(policy_cmd)
else:
    print("RUN_POLICY is False; command that would run:\n", " ".join(shlex.quote(str(c)) for c in policy_cmd))

In [ ]:
# What the agent found. Only four classes exist in its vocabulary, so an empty list is not 'healthy': it is 'none of these four'.
from collections import Counter
from dental_agent.benchmarks import runner as rn

pol = rn.read_jsonl(Path(RESULTS_DIR, "policy", POLICY_NAME, "findings.jsonl"))
if pol:
    answered = [r for r in pol if r["answered"]]
    classes = Counter(f["diagnosis"] for r in answered for f in r["findings"])
    print(f"{len(pol)} images | {len(answered)} with a valid final answer | {len(pol) - len(answered)} without")
    print(f"{sum(1 for r in answered if r['findings'])} images with at least one finding | {sum(len(r['findings']) for r in answered)} findings in total")
    print("findings by class:", dict(classes))
    print(f"tool calls per image: mean {sum(r['tool_calls'] for r in pol) / len(pol):.1f}, max {max(r['tool_calls'] for r in pol)}; "
          f"turns per image: mean {sum(r['turns'] for r in pol) / len(pol):.1f}; seconds per image: mean {sum(r['seconds'] for r in pol) / len(pol):.0f}")
else:
    print("No policy results yet.")

## 11. Phase 2a — Raw Qwen3.5-9B (the baseline row)
The untrained base model answers every question with the benchmark's **official prompt** and nothing else: image, question, options. Greedy, `MAX_NEW_TOKENS` tokens. A `<think>` block, if the chat template makes the model produce one, is removed before scoring and counted. This is both the paper's raw baseline and the answering model of Option A.

In [ ]:
def answer_cmd(system, *extra):
    return bench_cmd("answer", "--system-name", system, "--model-id", MODEL_ID, "--max-new-tokens", MAX_NEW_TOKENS,
                     "--existing-results", EXISTING_RESULTS, *RUNTIME_FLAGS, *LIMIT_FLAGS, *extra)

if RUN_RAW_BASELINE:
    run_cmd(answer_cmd(SYSTEM_RAW))

## 12. Phase 2b — Option A: the Same Raw Model + the Agent's Findings
Identical to 2a, except that each prompt starts with a fixed block (`findings_ctx_v1`) saying that an agent examined the image with zoom, contrast and tooth-localisation tools, that it only reports four conditions in FDI numbers, and what it found; the official prompt follows unchanged. The block states that the absence of a finding rules nothing out. All findings come from the cache of Phase 1; a missing image stops the step.

In [ ]:
if RUN_OPTION_A:
    run_cmd(answer_cmd(SYSTEM_A, "--findings-from", POLICY_NAME))

## 13. Phase 2c — Public Dental Baselines (same prompt, our judge)
Each model runs in its own process with its own processor, the official prompt and the same token budget; weights are loaded with `AutoModelForImageTextToText` and no remote code is executed. Attention/loader flags of the profile do not apply to them; only `EXTERNAL_PRECISION` does (settings cell). Pin a revision in `EXTERNAL_MODELS` to record exactly which weights were used; the commit hash actually loaded is saved in `answers/<name>/model_info.json`.

Gated models you have not accepted are **reported and skipped** (the gated-access audit, section 5), never silently replaced.

In [ ]:
if RUN_EXTERNAL_BASELINES:
    for name, spec in EXTERNAL_MODELS.items():
        if not spec["enabled"]:
            print(f"[SKIP] {name}: disabled in EXTERNAL_MODELS")
            continue
        if not GATE_STATUS.get(spec["repo"], False):
            print(f"[SKIP] {name}: {spec['repo']} is not accessible with this token (see the gated-access audit). Accept the terms, then re-run.")
            continue
        run_cmd(bench_cmd("answer", "--system-name", name, "--loader", "external", "--model-id", spec["repo"], "--model-revision", spec["revision"],
                          "--max-new-tokens", MAX_NEW_TOKENS, "--existing-results", EXISTING_RESULTS, "--precision", EXTERNAL_PRECISION, *LIMIT_FLAGS))

## 14. Phase 3 — The Judge (open-ended answers)
The benchmark's **official judge prompt** (8 worked examples, text only) scores each open-ended answer 0.0-1.0; up to 5 tries at temperature 0, 0.5, 1.0, ... and a reply that is not a valid score after 5 tries counts 0.0, exactly as in the official code. Two differences, both deliberate: an **API or transport error stops the run** (it resumes; it is never scored 0.0), and every judged row stores the judge identity, so a file judged by one model cannot be extended by another. Closed-ended questions need no judge.

Needs `JUDGE_MODEL` (settings cell) and the key in `JUDGE_KEY_ENV`.

In [ ]:
SYSTEMS_ALL = [SYSTEM_RAW, SYSTEM_A, *[k for k, v in EXTERNAL_MODELS.items() if v["enabled"]]]
SYSTEMS_DONE = [s for s in SYSTEMS_ALL if Path(RESULTS_DIR, "answers", s, "open.jsonl").is_file()]
print("systems with open-ended answers:", SYSTEMS_DONE)

if RUN_JUDGE:
    if not JUDGE_MODEL:
        raise ValueError("Set JUDGE_MODEL in the settings cell (section 6). There is no default: the judge is part of what the numbers mean.")
    if not os.environ.get(JUDGE_KEY_ENV):
        raise RuntimeError(f"{JUDGE_KEY_ENV} is not set")
    for system in SYSTEMS_DONE:
        run_cmd(bench_cmd("judge", "--system-name", system, "--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL,
                          "--judge-key-env", JUDGE_KEY_ENV, "--judge-max-tokens", JUDGE_MAX_TOKENS,
                          "--judge-send-temperature" if JUDGE_SEND_TEMPERATURE else "--no-judge-send-temperature",
                          "--workers", JUDGE_WORKERS, "--existing-results", "resume", *LIMIT_FLAGS))

## 15. Phase 4 — Scores and the Paper Table
Every number comes with a **95% interval from resampling radiographs** (questions of one image are correlated, so resampling questions would be too optimistic). Two scorings of the closed-ended answers are shown: the official one, where an unreadable answer gets a **random** option (seeded per question, so it is reproducible), and `[strict]`, where an unreadable answer is wrong. The gap between them tells you how much of a score is formatting rather than knowledge.

**Category columns count each question once** in every main category it belongs to. The benchmark's own scripts count a question whose category is a single label (for example `Jaw`) three times (closed) or twice (open) in the main-category columns, because several tallies share one key; Overall is not affected. Official-weighted values are saved in `mmoral_opg_scores.json` (`main_official_weighting`) if you need to compare with a paper that used the kit's own output.

**Published numbers of other papers** go in `published_numbers.json`, typed by hand, one object per row, each with its `source` and `judge`; a row without a source is refused. They are printed in separate italic rows and are not comparable to the measured rows unless the judge and protocol match:
```json
[{"name": "OralGPT-Omni-7B (own paper)", "part": "open", "judge": "name of the judge in that paper",
  "source": "paper, table, version", "Teeth": null, "Patho": null, "HisT": null, "Jaw": null, "SumRec": null, "Report": null, "Overall": 45.31}]
```
(`part` is `closed` or `open`; leave unknown categories `null`.)

In [ ]:
Path(PUBLISHED_NUMBERS).parent.mkdir(parents=True, exist_ok=True)
if not Path(PUBLISHED_NUMBERS).is_file():
    Path(PUBLISHED_NUMBERS).write_text("[]\n")
    print(f"[PUBLISHED] created an empty {PUBLISHED_NUMBERS}; add rows by hand (see the cell above).")

from IPython.display import Markdown, display
SYSTEMS_SCORED = [s for s in SYSTEMS_ALL if Path(RESULTS_DIR, "answers", s).is_dir()]
JUDGE_FLAGS = ["--judge-model", JUDGE_MODEL] if JUDGE_MODEL else []          # needed as soon as any system has open-ended answers
run_cmd(bench_cmd("score", "--systems", *SYSTEMS_SCORED, *JUDGE_FLAGS, "--published", PUBLISHED_NUMBERS))
display(Markdown(Path(RESULTS_DIR, "tables", "mmoral_opg.md").read_text()))

## 16. Phase 5 — Paired Comparisons (the claim)
The same questions answered by two systems; the mean difference (x100) with an interval that resamples radiographs. A `*` marks an interval that excludes zero.

- **Option A minus raw**: what the agent's findings add to the same model. This is the claim the paper can make.
- **Option A minus each public model** and **raw minus each public model**: where the agent-assisted system stands against models that were trained on MMOral-style data (they saw the benchmark's source distribution; ours did not).

In [ ]:
pairs = [(SYSTEM_A, SYSTEM_RAW)]
pairs += [(SYSTEM_A, n) for n in EXTERNAL_MODELS if n in SYSTEMS_SCORED]
pairs += [(SYSTEM_RAW, n) for n in EXTERNAL_MODELS if n in SYSTEMS_SCORED]
for a, b in pairs:
    if a in SYSTEMS_SCORED and b in SYSTEMS_SCORED:
        print("\n" + "=" * 100)
        run_cmd(bench_cmd("compare", "--a", a, "--b", b, *JUDGE_FLAGS))

## 17. Phase 6 — Answer-Quality Diagnostics
Numbers are only as good as the answers behind them. Per system: how many answers hit the token cap, how many have an unclosed `<think>` block (nothing to score), how many closed answers the official extractor cannot read (so they got a random option), and how many open answers are empty. A system with many of these needs inspection before anything is written about it.

In [ ]:
from dental_agent.benchmarks import mmoral_opg as mm, runner as rn
diag = []
for system in SYSTEMS_SCORED:
    base = Path(RESULTS_DIR, "answers", system)
    c, o = rn.read_jsonl(base / "closed.jsonl"), rn.read_jsonl(base / "open.jsonl")
    unreadable = (mm.score_closed(c)["unreadable"] if c else None)
    diag.append([system, len(c), len(o), sum(r["hit_token_cap"] for r in c + o), sum(r["unclosed_thinking"] for r in c + o),
                 sum(r["thinking_stripped"] for r in c + o), unreadable, sum(1 for r in o if not r["prediction"].strip()),
                 round(sum(r["new_tokens"] or 0 for r in c + o) / max(1, len(c) + len(o)), 1)])
print(tabulate(diag, headers=["system", "closed", "open", "hit token cap", "unclosed think", "think removed", "closed unreadable", "open empty", "mean new tokens"],
               tablefmt="github"))

In [ ]:
# Look at a few raw answers yourself, closed and open, for each system: what the model actually wrote, and what was scored.
import random as _random
from dental_agent.benchmarks import runner as rn
SHOW_N = 3
for system in SYSTEMS_SCORED:
    base = Path(RESULTS_DIR, "answers", system)
    for kind in ("closed", "open"):
        rows_ = rn.read_jsonl(base / f"{kind}.jsonl")
        for r in _random.Random(0).sample(rows_, min(SHOW_N, len(rows_))):
            print(f"--- {system} | {kind} #{r['index']} | {r['category']} | reference: {str(r['answer'])[:120]}")
            print("    prediction:", r["prediction"][:300].replace("\n", " "))

## 18. Phase 7 — Provenance
One file that says exactly what produced these numbers: git commit, package versions, dataset MD5s, the policy checkpoint and its adapter hashes, every system's settings, and the judge identity. The leakage screen report is included. Keep it with the paper's supplementary material.

In [ ]:
import importlib.metadata as md_
from dental_agent.benchmarks import runner as rn
def pkg(name):
    try:
        return md_.version(name)
    except md_.PackageNotFoundError:
        return None

systems_info = {}
for system in SYSTEMS_SCORED:
    base = Path(RESULTS_DIR, "answers", system)
    first = (rn.read_jsonl(base / "closed.jsonl") or rn.read_jsonl(base / "open.jsonl"))
    systems_info[system] = dict(settings=first[0]["settings"] if first else None,
                                model_info=json.loads((base / "model_info.json").read_text()) if (base / "model_info.json").is_file() else None)
judge_ids = {}
for jf in sorted(Path(RESULTS_DIR, "judge").glob("*.jsonl")) if Path(RESULTS_DIR, "judge").is_dir() else []:
    rows_ = rn.read_jsonl(jf)
    judge_ids[jf.name] = {k: rows_[0][k] for k in rows_[0] if k.startswith("judge") or k in ("max_completion_tokens", "send_temperature")} if rows_ else None
provenance = dict(
    git_commit=subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    git_dirty=bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip()),
    profile=HARDWARE_PROFILE, tag=TAG, runtime_flags=RUNTIME_FLAGS, max_new_tokens=MAX_NEW_TOKENS,
    packages={p: pkg(p) for p in ("torch", "transformers", "peft", "huggingface_hub", "openai", "qwen-vl-utils", "accelerate")},
    gpu=[(d["name"], round(d["gib"], 1)) for d in DEVICES], benchmark=manifest, policy=dict(name=POLICY["name"], source=POLICY["source"], meta=POLICY["meta"]),
    policy_settings=(pol[0]["settings"] if pol else None), systems=systems_info, judges=judge_ids,
    leakage_screen_vs_dentex=(json.loads(Path(BENCH_DIR, "leakage_dentex.json").read_text()) if Path(BENCH_DIR, "leakage_dentex.json").is_file() else None),
)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR, "PROVENANCE.json").write_text(json.dumps(provenance, indent=2, default=str))
print("written:", Path(RESULTS_DIR, "PROVENANCE.json"), "| git dirty:", provenance["git_dirty"])

## 19. Phase 8 — Sync to the Hub
Copies the results folder (answers, judge files, policy findings, tables, provenance; **no images**) to `external/mmoral_opg/<profile tag>/` in the existing results repository, and restores it on a fresh server without overwriting anything that already exists locally. The repository is never created by code. Do this before the rented server goes away.

In [ ]:
SYNC_DIRECTION = None            # "up" to upload, "down" to restore a previous session's results, None to skip
if SYNC_DIRECTION:
    run_cmd(bench_cmd("sync", "--direction", SYNC_DIRECTION, "--tag", TAG, "--repo", EVAL_RESULTS_REPO,
                      *(["--repo-type", EVAL_RESULTS_REPO_TYPE] if EVAL_RESULTS_REPO_TYPE else [])))
else:
    print("SYNC_DIRECTION is None: nothing synced.")

## 20. How to Read and Report the Result

**What the table can support**
- *Option A minus raw* (Phase 5) is the clean, same-model, same-judge, same-prompt statement of what the agent adds. Report it with its interval.
- Rows of public models answer a different question (what a model trained on MMOral-style data does here); say so in the paper.
- Copied numbers of other papers stay in their own rows with their judge named.

**Limitations paragraph (facts this notebook establishes)**
- All earlier evaluation is on the 50 DENTEX validation images; this adds 100 radiographs from another source, scored by a judge whose identity changes the numbers. Say which judge.
- The policy emits findings for four classes only; MMOral also asks about teeth counts, past treatments, jaw and recommendations. The agent-assisted system therefore uses a two-stage design (agent, then answering model); the gain is limited to what the four-class findings can inform.
- The grounder and all policy training data are DENTEX; the leakage screen (section 8) found/did not find overlap, as printed there. It does not cover crops, rotations or flips.
- Closed-ended scoring follows the official extractor, which falls back to a random option for unreadable answers; the strict score and the unreadable count are reported next to it.

**If the numbers are low:** switch to Option B (the agent answers in free text under a new prompt). That is a different system with its own prompt, to be built and frozen before running; do not tune Option A's findings block after seeing scores.

## 21. Troubleshooting
| Symptom | Where to look |
|---|---|
| `prepare` stops on MD5 | the Hub file changed or the download is damaged; inspect, then `--allow-md5-mismatch` (recorded in the manifest) |
| `NOT ACCEPTED` in the gated audit | open the model page with the `HF_TOKEN` account and click Agree |
| policy: "adapter ... does not exist" | section 9 output; the exact name must exist (K, harness mode, stage, tag) |
| policy or answer: refuses to resume, "different settings" | a setting changed; `EXISTING_RESULTS = "overwrite"` for that system only, or restore the old value |
| judge: rejects `temperature` | `JUDGE_SEND_TEMPERATURE = False` (recorded in the judge identity) |
| judge: "judged with different settings" | a different judge was used earlier for that system; use one judge for the whole table |
| many "hit token cap" or "unclosed think" | the answering model rambles or thinks past the budget; the budget is not lowered, inspect the raw answers (Phase 6) |
| many "closed unreadable" | the model does not answer with a letter; read the raw answers; the strict score shows the effect |
| external model fails to load | `transformers` version vs the model's architecture; the error names the class |